# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madaraf/Starter-Notebooks-Assignment-Flyrank-/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:

%pip -q install duckdb huggingface_hub scikit-learn pandas



In [3]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Base paths
REL = 'hf://datasets/FlyRank/internship-warehouse'

Paste your Hugging Face READ token (hf_...): ··········


In [4]:
# Feature month (March 2026)
MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Outcome month (April 2026)
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

# Dimension table
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

In [5]:
features_df = con.sql(f"""
    WITH page_month AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions)                        AS impressions_month,
            SUM(gsc_clicks)                             AS clicks_month,
            AVG(gsc_avg_position)                       AS avg_position_month,
            COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions
        FROM {MONTH}
        GROUP BY content_hash_id, client_hash_id
        HAVING SUM(gsc_impressions) >= 100
    )
    SELECT
        p.content_hash_id,
        p.client_hash_id,
        p.impressions_month,
        p.clicks_month,
        ROUND(p.clicks_month * 1.0 / NULLIF(p.impressions_month, 0), 4) AS ctr_month,
        p.avg_position_month,
        p.days_with_impressions,
        DATE_DIFF('day', d.content_created_date, DATE '2026-03-31')     AS content_age_days
    FROM page_month p
    LEFT JOIN {DIM_CONTENT} d
        ON p.content_hash_id = d.content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
# Target window aggregation
april_impressions = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_april
    FROM {APRIL}
    GROUP BY content_hash_id
""").df()

# Merge and calculate label
labeled_df = features_df.merge(april_impressions, on="content_hash_id", how="inner")
labeled_df["pct_change"] = (
    (labeled_df["impressions_april"] - labeled_df["impressions_month"])
    / labeled_df["impressions_month"]
)
labeled_df["is_declining_forward_label"] = (labeled_df["pct_change"] <= -0.20).astype(int)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [7]:
# ============================================================
# TASK 1: Method choice — Decision Tree + Random Forest (no Logistic Regression)
# ============================================================
# Logistic Regression skipped on purpose: Week 4's staleness signal is
# non-monotonic (rises to a peak at 91-180 days, then falls for older pages).
# Logistic regression assumes a roughly straight-line relationship per
# feature, so it structurally can't represent a "rise then fall" shape.
# A tree can split content_age_days multiple times at different thresholds,
# so it CAN carve out that middle bump directly.

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

honest_features = ["impressions_month", "ctr_month", "avg_position_month",
                    "days_with_impressions", "content_age_days"]


I chose Decision Tree and Random Forest, and deliberately skipped Logistic Regression. My Week 4 signal audit found that staleness (content_age_days) has a non-monotonic relationship with decline — it rises to a peak at 91-180 days, then falls for older pages. Logistic Regression assumes a roughly straight-line relationship between each feature and the outcome, so it structurally cannot represent a "rise then fall" shape. A Decision Tree can split content_age_days at multiple thresholds, letting it carve out that middle bump directly — and a Random Forest inherits this same ability while also being more stable than a single tree. I'm training both to compare, rather than assuming one wins in advance.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [37]:
# ============================================================
# TASK 2: Split design — client-grouped split
# ============================================================
# Grouped by client_hash_id so a client's pages are ENTIRELY in train
# OR ENTIRELY in test, never split across both. This blocks leakage that
# would survive even WITHOUT client_id as a feature (client-specific
# patterns show up through other correlated features, not just an ID).
# No separate time-based split needed on top of this — the time gap
# (March features -> April label) is already built into the label itself.

from sklearn.model_selection import GroupShuffleSplit

X = labeled_df[honest_features].fillna(0)
y = labeled_df["is_declining_forward_label"]
groups = labeled_df["client_hash_id"]

splitter = GroupShuffleSplit(test_size=0.25, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients  = set(groups.iloc[test_idx])
print(f"Train rows: {len(X_train):,}   Test rows: {len(X_test):,}")
print(f"Train clients: {len(train_clients)}   Test clients: {len(test_clients)}")
print(f"Client overlap between train and test (must be 0): {len(train_clients & test_clients)}")

Train rows: 93,085   Test rows: 8,356
Train clients: 33   Test clients: 11
Client overlap between train and test (must be 0): 0


I used a client-grouped split (GroupShuffleSplit on client_hash_id), so that all of a given client's pages fall entirely into train OR entirely into test, never split across both. This matters because client-specific patterns (writing style, typical CTR, content cadence) can leak into the model through other correlated features even without client_id itself being used as a feature — a plain random row split wouldn't catch this. I did not add a separate time-based split on top of this: the time gap is already built into the label itself (is_declining_forward_label compares March features against a separate, later April outcome window from Week 3), so a second time split would be redundant. Result: 93,085 train rows across 33 clients, 8,356 test rows across 11 clients, with 0 client overlap between train and test — confirming the split is clean.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [38]:
import pandas as pd

# Load the baseline results generated by the old notebook
queue = pd.read_csv("baseline_action_score.csv")

In [44]:
# ============================================================
# TASK 3: Train + compare vs baseline — SAME split, SAME metric
# ============================================================
# Fairness note: baseline uses 2 signals, models use 5 features — NOT
# equalized on purpose (see explanation above). What IS held identical:
# the exact same test rows, and the exact same Precision@50 metric.

import numpy as np
import pandas as pd

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(y_true)[order[:k]]
    return topk.mean()

# --- Decision Tree ---
tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)
tree_scores = tree.predict_proba(X_test)[:, 1]
tree_p50 = precision_at_k(y_test.values, tree_scores, 50)

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=6,
                             class_weight="balanced_subsample",
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
rf_p50 = precision_at_k(y_test.values, rf_scores, 50)

# --- Baseline (Week 4 rule), scored on the SAME test rows ---
test_content_ids = labeled_df.iloc[test_idx]["content_hash_id"].values
baseline_lookup = queue.set_index("content_hash_id")["baseline_action_score"]
baseline_test_scores = baseline_lookup.reindex(test_content_ids).fillna(0).values
baseline_p50 = precision_at_k(y_test.values, baseline_test_scores, 50)

print(f"Baseline rule (2 signals)     Precision@50: {baseline_p50:.3f}")
print(f"Decision Tree (5 features)    Precision@50: {tree_p50:.3f}")
print(f"Random Forest (5 features)    Precision@50: {rf_p50:.3f}")

comparison_table = pd.DataFrame({
    "method": ["baseline_rule", "decision_tree", "random_forest"],
    "feature_count": [2, 5, 5],
    "precision_at_50": [baseline_p50, tree_p50, rf_p50]
})
comparison_table

Baseline rule (2 signals)     Precision@50: 0.500
Decision Tree (5 features)    Precision@50: 0.320
Random Forest (5 features)    Precision@50: 0.640


,method,feature_count,precision_at_50
0,baseline_rule,2,0.50
1,decision_tree,5,0.32
2,random_forest,5,0.64


In [42]:
# Diagnostic: is the tree's weak score specific to depth=4, or a general instability?
# Not cherry-picking a "winning" depth — just checking the shape of the pattern.
print("Decision Tree Precision@50 across several depths (diagnostic only):")
for depth in [2, 3, 4, 5, 6, 8]:
    t = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42)
    t.fit(X_train, y_train)
    p50 = precision_at_k(y_test.values, t.predict_proba(X_test)[:, 1], 50)
    print(f"  max_depth={depth}:  Precision@50 = {p50:.3f}")

# Also worth knowing: how big/small is the test set really?
print(f"\nTest set: {len(X_test)} rows across {labeled_df.iloc[test_idx]['client_hash_id'].nunique()} clients")
print("(Precision@50 on a small client-holdout test set can swing a lot between runs —")
print(" a single number here is informative, not a guaranteed stable estimate.)")

Decision Tree Precision@50 across several depths (diagnostic only):
  max_depth=2:  Precision@50 = 0.660
  max_depth=3:  Precision@50 = 0.860
  max_depth=4:  Precision@50 = 0.320
  max_depth=5:  Precision@50 = 0.560
  max_depth=6:  Precision@50 = 0.340
  max_depth=8:  Precision@50 = 0.400

Test set: 8356 rows across 11 clients
(Precision@50 on a small client-holdout test set can swing a lot between runs —
 a single number here is informative, not a guaranteed stable estimate.)


Random Forest beats the baseline (0.640 vs 0.500, ~28% relative improvement), on the same held-out clients and the same Precision@50 metric — this satisfies the requirement that the model must beat the baseline. Decision Tree underperforms the baseline (0.320), which I investigated rather than hid: a depth-by-depth diagnostic (max_depth 2 through 8) showed scores bouncing unpredictably (0.660 → 0.860 → 0.320 → 0.560 → 0.340 → 0.400) with no clean trend. The most likely explanation is that the test set contains only 11 clients — small enough that one or two clients' pages can dominate which rows land in the top-50, making a single tree's Precision@50 unstable run to run. I'm reporting my originally-specified max_depth=4 result (0.320) rather than substituting a better-looking depth after seeing this diagnostic, since doing so would be cherry-picking noise rather than a genuine improvement. Note on fairness: the baseline uses 2 signals and the models use 5 features — not equalized on purpose, since forcing the models down to 2 features would hide the actual advantage ML can offer (combining more signals than a hand rule can manage). What IS held identical across all three: the exact same test rows and the exact same Precision@50 metric.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [45]:
# Build an error-analysis frame on the test set
test_results = labeled_df.iloc[test_idx][["content_hash_id", "content_age_days"]].copy()
test_results["y_true"] = y_test.values
test_results["rf_score"] = rf_scores
test_results["rf_pred"] = (rf_scores >= 0.5).astype(int)
test_results["correct"] = (test_results["y_true"] == test_results["rf_pred"]).astype(int)

def age_bucket(days):
    if days <= 90: return "fresh_0_90"
    elif days <= 180: return "aging_91_180"
    elif days <= 365: return "stale_181_365"
    else: return "very_stale_365plus"

test_results["age_bucket"] = test_results["content_age_days"].apply(age_bucket)

error_by_age = test_results.groupby("age_bucket").agg(
    n=("correct", "size"),
    accuracy=("correct", "mean"),
    true_decline_rate=("y_true", "mean"),
    predicted_decline_rate=("rf_pred", "mean")
).round(3)

print(error_by_age)

                       n  accuracy  true_decline_rate  predicted_decline_rate
age_bucket                                                                   
aging_91_180         438     0.489              0.372                   0.705
fresh_0_90          6023     0.616              0.365                   0.332
stale_181_365        249     0.526              0.751                   0.582
very_stale_365plus  1646     0.327              0.911                   0.277


In [46]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    rf, X_test, y_test, n_repeats=20, random_state=42, scoring="roc_auc"
)

importance_df = pd.DataFrame({
    "feature": honest_features,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values("importance_mean", ascending=False)

print(importance_df)

                 feature  importance_mean  importance_std
1              ctr_month         0.062049        0.004709
0      impressions_month         0.022400        0.001151
3  days_with_impressions         0.005229        0.002290
2     avg_position_month        -0.008982        0.001428
4       content_age_days        -0.019091        0.004730


Where the model is wrong: error broken down by content age bucket shows the Random Forest is least accurate on the oldest pages — very_stale_365plus has only 32.7% accuracy, the lowest of all four buckets, consistent with the non-monotonic staleness pattern found in Week 4. Specifically, the model under-flags very old pages: 91.1% of them actually declined in the test set, but the model only predicted decline for 27.7% of them. This suggests the age–decline relationship learned from the 33 training clients (where very-old pages had a lower decline rate than mid-aged pages) did not generalize to these 11 held-out clients, where very-old pages turned out to decline heavily. The aging_91_180 (n=438) and stale_181_365 (n=249) buckets are small enough that their numbers deserve less confidence than the 6,023-row fresh_0_90 bucket.

What the model leans on: permutation importance shows ctr_month as by far the strongest driver (0.062), followed by impressions_month (0.022) and days_with_impressions (0.005). avg_position_month (-0.009) and content_age_days (-0.019) came back slightly negative — meaning the model barely uses them at all; the negative sign reflects noise around zero rather than the features being actively harmful. This lines up closely with my Week 4 signal verdicts: CTR-vs-position was CONFIRMED, and the model leans on it most; staleness was MIXED, and the model barely relies on it. My independent, hand-checked signal audit and the model's own learned behavior are telling the same story from two different angles.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.